# Autoencoders using PyTorch with progressively complex examples.

1. **Image Denoising** - Removing noise from MNIST and Fashion-MNIST images
2. **Dimensionality Reduction** - Visualizing latent space representations
3. **Convolutional Autoencoders** - Applying Modern CNN-based architectures
4. **Experiments & Analysis** - Systematic exploration of design choices

**Learning Objectives:**
- Understand encoder-decoder architecture and the bottleneck concept
- Implement autoencoders from scratch in PyTorch
- Visualize and interpret latent space representations
- Apply autoencoders to real-world tasks
- Analyze the effect of hyperparameters on performance

# 2. Import all necessary libraries.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

from sklearn.manifold import TSNE # for t-SNE visualization (Stochastic Neighbor Embedding)
import seaborn as sns

In [ ]:
# Utilities
import time
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Device configuration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 3. Helper Functions

We'll create reusable functions for common tasks throughout the workshop.

In [ ]:
def add_noise(images, noise_factor=0.3):
    """Noisy images clipped to [0, 1]
    """
    noise = torch.randn_like(images) * noise_factor
    noisy_images = images + noise
    return torch.clip(noisy_images, 0., 1.) # Clip to [0, 1]


def visualize_reconstructions(original, noisy, reconstructed, n=10, dataset_name=""):
    fig, axes = plt.subplots(3, n, figsize=(20, 6))
    
    for i in range(n):
        # Original
        axes[0, i].imshow(original[i].cpu().squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Original', fontsize=12, fontweight='bold')
        
        # Noisy
        axes[1, i].imshow(noisy[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Noisy Input', fontsize=12, fontweight='bold')
        
        # Reconstructed
        axes[2, i].imshow(reconstructed[i].cpu().squeeze(), cmap='gray')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_title('Reconstructed', fontsize=12, fontweight='bold')
    
    plt.suptitle(f'{dataset_name} - Denoising Results', fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()


def plot_training_curves(train_losses, val_losses=None, title="Training Progress"):
    plt.figure(figsize=(10, 5))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    if val_losses:
        plt.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def visualize_bottleneck(model, data_loader, device, n_samples=1000):
    model.eval()
    bottleneck_activations = []
    
    with torch.no_grad():
        for images, _ in data_loader:
            images = images.view(images.size(0), -1).to(device)
            # Get bottleneck representation
            encoded = model.encoder(images)
            bottleneck_activations.append(encoded.cpu().numpy())
            
            if len(bottleneck_activations) * images.size(0) >= n_samples:
                break
    
    bottleneck_activations = np.concatenate(bottleneck_activations, axis=0)[:n_samples]
    
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram of all activations
    axes[0].hist(bottleneck_activations.flatten(), bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Activation Value', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Distribution of Bottleneck Activations', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Heatmap of activations for first 100 samples
    im = axes[1].imshow(bottleneck_activations[:100].T, aspect='auto', cmap='viridis')
    axes[1].set_xlabel('Sample Index', fontsize=12)
    axes[1].set_ylabel('Bottleneck Dimension', fontsize=12)
    axes[1].set_title('Bottleneck Activations Heatmap', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=axes[1])
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"Bottleneck Statistics (n={n_samples} samples):")
    print(f"  Shape: {bottleneck_activations.shape}")
    print(f"  Mean: {bottleneck_activations.mean():.4f}")
    print(f"  Std: {bottleneck_activations.std():.4f}")
    print(f"  Min: {bottleneck_activations.min():.4f}")
    print(f"  Max: {bottleneck_activations.max():.4f}")


def calculate_metrics(original, reconstructed):
    """
    Calculate reconstruction quality metrics.
    
    Args:
        original: Original images
        reconstructed: Reconstructed images
    
    Returns:
        Dictionary with MSE and PSNR
    """
    mse = torch.mean((original - reconstructed) ** 2).item()
    
    # PSNR (Peak Signal-to-Noise Ratio)
    # Assumes pixel values in [0, 1]
    if mse > 0:
        psnr = 20 * np.log10(1.0 / np.sqrt(mse))
    else:
        psnr = float('inf') # PSNR: Peak Signal-to-Noise Ratio (dB) measurement of the quality of an image or signal
    
    return {'MSE': mse, 'PSNR': psnr}

print("Helper functions defined successfully!")

## 4. Example 1: Image Denoising with MNIST

### 4.1 Data Preparation

We'll load the MNIST dataset and create noisy versions for training our denoising autoencoder.

In [ ]:
# Hyperparameters
batch_size = 128
noise_factor = 0.3

# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0, 1] range
])

train_dataset = torchvision.datasets.MNIST(root='./data',train=True,transform=transform,download=True)

test_dataset = torchvision.datasets.MNIST(root='./data',train=False,transform=transform,download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Batch size: {batch_size}")

In [ ]:
# Visualize clean MNIST samples
fig, axes = plt.subplots(2, 10, figsize=(20, 4))

for i in range(10):
    img, label = train_dataset[i]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'Label: {label}', fontsize=10)
    axes[0, i].axis('off')
    
    # Add noise
    noisy_img = add_noise(img.unsqueeze(0), noise_factor).squeeze()
    axes[1, i].imshow(noisy_img, cmap='gray')
    axes[1, i].set_title(f'Noisy', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Noisy', fontsize=12, fontweight='bold')
plt.suptitle('MNIST Dataset: Clean vs Noisy Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Model Architecture

We'll build a fully-connected autoencoder with the following architecture:

**Encoder:**
- Input: 784 (28×28 flattened)
- Hidden: 784 → 128 → 64 → 32 (bottleneck)

**Decoder:**
- Bottleneck: 32
- Hidden: 32 → 64 → 128 → 784
- Output: 784 (reconstructed 28×28 image)

**Key Design Choices:**
- **Activation**: ReLU for hidden layers (introduces non-linearity)
- **Output Activation**: Sigmoid (ensures output in [0, 1] range)
- **Bottleneck Size**: 32 dimensions (24x compression from 784)

In [ ]:
class Autoencoder(nn.Module):
    """
    Fully-connected Autoencoder for image denoising.
    
    Architecture:
        Encoder: 784 → 128 → 64 → 32
        Decoder: 32 → 64 → 128 → 784
    """
    def __init__(self, input_dim=784, hidden_dims=[128, 64], bottleneck_dim=32):
        super(Autoencoder, self).__init__()
        
        # Encoder layers
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),  # 784 → 128
            nn.ReLU(True),
            nn.Linear(hidden_dims[0], hidden_dims[1]),  # 128 → 64
            nn.ReLU(True),
            nn.Linear(hidden_dims[1], bottleneck_dim),  # 64 → 32 (bottleneck)
            nn.ReLU(True)
        )
        
        # Decoder layers (mirror of encoder)
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dims[1]),  # 32 → 64
            nn.ReLU(True),
            nn.Linear(hidden_dims[1], hidden_dims[0]),  # 64 → 128
            nn.ReLU(True),
            nn.Linear(hidden_dims[0], input_dim),  # 128 → 784
            nn.Sigmoid()  # Output in [0, 1] range
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        """Get bottleneck representation."""
        return self.encoder(x)
    
    def decode(self, z):
        """Reconstruct from bottleneck."""
        return self.decoder(z)


# Initialize model
model = Autoencoder().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nCompression ratio: {784/32:.1f}x")

### 4.3 Training the Autoencoder

**Training Strategy:**
1. Add noise to clean images
2. Feed noisy images to the model
3. Compare reconstruction with **clean** images (not noisy)
4. Minimize reconstruction error using MSE loss
5. Update weights via backpropagation

**Loss Function:** Mean Squared Error (MSE)
$$
\mathcal{L} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2
$$

**Optimizer:** Adam with learning rate 0.001

In [ ]:
# Training configuration
num_epochs = 20
learning_rate = 0.001

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training history
train_losses = []
test_losses = []

print(f"Starting training for {num_epochs} epochs...")
print(f"Noise factor: {noise_factor}")
print(f"Learning rate: {learning_rate}")

start_time = time.time()

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for batch_idx, (images, _) in enumerate(train_loader):
        # Flatten images and move to device
        images = images.view(images.size(0), -1).to(device)
        
        # Add noise to create corrupted input
        noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_factor)
        noisy_images = noisy_images.view(noisy_images.size(0), -1)
        
        # Forward pass: reconstruct from noisy input
        reconstructed = model(noisy_images)
        
        # Loss: compare reconstruction with CLEAN images
        loss = criterion(reconstructed, images)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Average training loss for epoch
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Evaluation phase
    model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.view(images.size(0), -1).to(device)
            noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_factor)
            noisy_images = noisy_images.view(noisy_images.size(0), -1)
            
            reconstructed = model(noisy_images)
            loss = criterion(reconstructed, images)
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(test_loader)
    test_losses.append(avg_test_loss)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Loss: {avg_train_loss:.6f} | "
              f"Test Loss: {avg_test_loss:.6f}")

elapsed_time = time.time() - start_time
print(f"Training completed in {elapsed_time:.2f} seconds")
print(f"Final Train Loss: {train_losses[-1]:.6f}")
print(f"Final Test Loss: {test_losses[-1]:.6f}")

### 4.4 Results Visualization

Let's visualize the training progress and reconstruction quality.

In [ ]:
# Plot training curves
plot_training_curves(train_losses, test_losses, "MNIST Denoising Autoencoder - Training Progress")

In [ ]:
# Visualize reconstructions on test set
model.eval()

with torch.no_grad():
    # Get a batch of test images
    test_images, _ = next(iter(test_loader))
    test_images = test_images.to(device)
    
    # Add noise
    noisy_test = add_noise(test_images, noise_factor)
    
    # Reconstruct
    noisy_flat = noisy_test.view(noisy_test.size(0), -1)
    reconstructed_flat = model(noisy_flat)
    reconstructed = reconstructed_flat.view(-1, 1, 28, 28)
    
    # Visualize
    visualize_reconstructions(test_images, noisy_test, reconstructed, n=10, dataset_name="MNIST")

## 5. Example 2: Fashion-MNIST Denoising

### 5.1 Why Fashion-MNIST?

Fashion-MNIST is a more challenging dataset than MNIST:
- **More complex patterns**: Clothing items vs simple digits
- **Higher intra-class variance**: More variation within each category
- **Better real-world analog**: Closer to practical image tasks

Let's apply the same autoencoder architecture and see how it performs!

In [ ]:
# Load Fashion-MNIST dataset
fashion_train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

fashion_test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

fashion_train_loader = DataLoader(fashion_train_dataset, batch_size=batch_size, shuffle=True)
fashion_test_loader = DataLoader(fashion_test_dataset, batch_size=batch_size, shuffle=False)

# Class names for Fashion-MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training samples: {len(fashion_train_dataset)}")
print(f"Test samples: {len(fashion_test_dataset)}")
print(f"Classes: {class_names}")

In [ ]:
# Visualize Fashion-MNIST samples
fig, axes = plt.subplots(2, 10, figsize=(20, 4))

for i in range(10):
    img, label = fashion_train_dataset[i]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'{class_names[label]}', fontsize=9)
    axes[0, i].axis('off')
    
    # Add noise
    noisy_img = add_noise(img.unsqueeze(0), noise_factor).squeeze()
    axes[1, i].imshow(noisy_img, cmap='gray')
    axes[1, i].set_title(f'Noisy', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Noisy', fontsize=12, fontweight='bold')
plt.suptitle('Fashion-MNIST Dataset: Clean vs Noisy Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Training on Fashion-MNIST

We'll use the same architecture and hyperparameters to demonstrate generalization.

In [ ]:
# Initialize new model for Fashion-MNIST, trained on noisy images

fashion_model = Autoencoder().to(device)
fashion_criterion = nn.MSELoss()
fashion_optimizer = optim.Adam(fashion_model.parameters(), lr=learning_rate)

# Training history
fashion_train_losses = []
fashion_test_losses = []

print(f"Training Fashion-MNIST autoencoder for {num_epochs} epochs...")
print("-" * 60)

start_time = time.time()

for epoch in range(num_epochs):
    # Training phase
    fashion_model.train()
    train_loss = 0.0
    
    for batch_idx, (images, _) in enumerate(fashion_train_loader):
        images = images.view(images.size(0), -1).to(device)
        noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_factor)
        noisy_images = noisy_images.view(noisy_images.size(0), -1)
        
        reconstructed = fashion_model(noisy_images)
        loss = fashion_criterion(reconstructed, images)
        
        fashion_optimizer.zero_grad()
        loss.backward()
        fashion_optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(fashion_train_loader)
    fashion_train_losses.append(avg_train_loss)
    
    # Evaluation phase
    fashion_model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for images, _ in fashion_test_loader:
            images = images.view(images.size(0), -1).to(device)
            noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_factor)
            noisy_images = noisy_images.view(noisy_images.size(0), -1)
            
            reconstructed = fashion_model(noisy_images)
            loss = fashion_criterion(reconstructed, images)
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(fashion_test_loader)
    fashion_test_losses.append(avg_test_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Loss: {avg_train_loss:.6f} | "
              f"Test Loss: {avg_test_loss:.6f}")

elapsed_time = time.time() - start_time
print("-" * 60)
print(f"Training completed in {elapsed_time:.2f} seconds")
print(f"Final Train Loss: {fashion_train_losses[-1]:.6f}")
print(f"Final Test Loss: {fashion_test_losses[-1]:.6f}")

### 5.3 Fashion-MNIST Results

In [ ]:
# Plot training curves
plot_training_curves(fashion_train_losses, fashion_test_losses, 
                    "Fashion-MNIST Denoising Autoencoder - Training Progress")

In [ ]:
# Visualize reconstructions
fashion_model.eval()

with torch.no_grad():
    fashion_test_images, fashion_labels = next(iter(fashion_test_loader))
    fashion_test_images = fashion_test_images.to(device)
    
    fashion_noisy_test = add_noise(fashion_test_images, noise_factor)
    fashion_noisy_flat = fashion_noisy_test.view(fashion_noisy_test.size(0), -1)
    fashion_reconstructed_flat = fashion_model(fashion_noisy_flat)
    fashion_reconstructed = fashion_reconstructed_flat.view(-1, 1, 28, 28)
    
    visualize_reconstructions(fashion_test_images, fashion_noisy_test, fashion_reconstructed, 
                            n=10, dataset_name="Fashion-MNIST")

### 5.4 Comparison: MNIST vs Fashion-MNIST

Let's compare the performance on both datasets.

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MNIST
epochs = range(1, len(train_losses) + 1)
axes[0].plot(epochs, train_losses, 'b-', label='Train', linewidth=2)
axes[0].plot(epochs, test_losses, 'r-', label='Test', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('MNIST', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Fashion-MNIST
axes[1].plot(epochs, fashion_train_losses, 'b-', label='Train', linewidth=2)
axes[1].plot(epochs, fashion_test_losses, 'r-', label='Test', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss (MSE)', fontsize=12)
axes[1].set_title('Fashion-MNIST', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Comparison: MNIST vs Fashion-MNIST', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nFinal Test Loss Comparison:")
print(f"  MNIST:         {test_losses[-1]:.6f}")
print(f"  Fashion-MNIST: {fashion_test_losses[-1]:.6f}")
print(f"  Difference:    {abs(fashion_test_losses[-1] - test_losses[-1]):.6f}")
print(f"\nObservation: Fashion-MNIST typically has higher loss due to increased complexity.")

## 6. Dimensionality Reduction & Latent Space Visualization

### 6.1 Understanding the Latent Space

The bottleneck layer creates a compressed representation of the input. 

By reducing it to 2 dimensions, we can visualize how the autoencoder organizes different classes in latent space.

**Key Questions:**
- Do similar items cluster together?
- Are classes separable in latent space?
- What does the autoencoder learn without labels?

In [ ]:
class Autoencoder2D(nn.Module):
    """
    Autoencoder with 2D bottleneck for visualization.
    
    Architecture:
        Encoder: 784 → 256 → 128 → 2 (bottleneck)
        Decoder: 2 → 128 → 256 → 784
    """
    def __init__(self, input_dim=784, hidden_dims=[256, 128], bottleneck_dim=2):
        super(Autoencoder2D, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(True),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(True),
            nn.Linear(hidden_dims[1], bottleneck_dim),  # 2D bottleneck
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dims[1]),
            nn.ReLU(True),
            nn.Linear(hidden_dims[1], hidden_dims[0]),
            nn.ReLU(True),
            nn.Linear(hidden_dims[0], input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)


# Initialize 2D model
model_2d = Autoencoder2D().to(device)
print(model_2d)
print(f"\nBottleneck dimension: 2 (for visualization)")
print(f"Compression ratio: {784/2:.1f}x")

### 6.2 Training 2D Autoencoder

We'll train on Fashion-MNIST for more interesting visualizations.

In [ ]:
# Training configuration
num_epochs_2d = 30  # More epochs for 2D bottleneck
criterion_2d = nn.MSELoss()
optimizer_2d = optim.Adam(model_2d.parameters(), lr=0.001)

train_losses_2d = []
test_losses_2d = []

print(f"Training 2D autoencoder for {num_epochs_2d} epochs...")
start_time = time.time()

for epoch in range(num_epochs_2d):
    model_2d.train()
    train_loss = 0.0
    
    for images, _ in fashion_train_loader:
        images = images.view(images.size(0), -1).to(device)
        
        reconstructed = model_2d(images)
        loss = criterion_2d(reconstructed, images)
        
        optimizer_2d.zero_grad()
        loss.backward()
        optimizer_2d.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(fashion_train_loader)
    train_losses_2d.append(avg_train_loss)
    
    # Evaluation
    model_2d.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for images, _ in fashion_test_loader:
            images = images.view(images.size(0), -1).to(device)
            reconstructed = model_2d(images)
            loss = criterion_2d(reconstructed, images)
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(fashion_test_loader)
    test_losses_2d.append(avg_test_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_2d}] | "
              f"Train Loss: {avg_train_loss:.6f} | "
              f"Test Loss: {avg_test_loss:.6f}")

elapsed_time = time.time() - start_time
print("-" * 60)
print(f"Training completed in {elapsed_time:.2f} seconds")

In [ ]:
# Plot training curves
plot_training_curves(train_losses_2d, test_losses_2d, 
                    "2D Autoencoder - Training Progress")

### 6.3 Latent Space Visualization

Now let's visualize how the 2D autoencoder organizes Fashion-MNIST classes in latent space.

In [ ]:
# Extract latent representations for entire test set
model_2d.eval()

latent_vectors = []
labels_list = []

with torch.no_grad():
    for images, labels in fashion_test_loader:
        images = images.view(images.size(0), -1).to(device)
        encoded = model_2d.encode(images)
        latent_vectors.append(encoded.cpu().numpy())
        labels_list.append(labels.numpy())

latent_vectors = np.concatenate(latent_vectors, axis=0)
labels_array = np.concatenate(labels_list, axis=0)

print(f"Latent space shape: {latent_vectors.shape}")
print(f"Labels shape: {labels_array.shape}")

In [ ]:
# Visualize latent space with class labels
plt.figure(figsize=(12, 10))

# Create scatter plot for each class
for i, class_name in enumerate(class_names):
    mask = labels_array == i
    plt.scatter(latent_vectors[mask, 0], latent_vectors[mask, 1], 
                label=class_name, alpha=0.6, s=10)

plt.xlabel('Latent Dimension 1', fontsize=12)
plt.ylabel('Latent Dimension 2', fontsize=12)
plt.title('2D Latent Space Visualization - Fashion-MNIST', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- The autoencoder learned meaningful representations WITHOUT labels")
print("- Some overlap indicates shared visual features between classes")

### 6.4 Latent Space Interpolation

We can interpolate between two points in latent space and decode to see smooth transitions.

In [ ]:
# Select two random test images from different classes
model_2d.eval()

# Get images from different classes
class_a, class_b = 0, 8  # T-shirt and Bag
idx_a = np.where(labels_array == class_a)[0][0]
idx_b = np.where(labels_array == class_b)[0][0]

with torch.no_grad():
    # Get images
    img_a = fashion_test_dataset[idx_a][0].view(1, -1).to(device)
    img_b = fashion_test_dataset[idx_b][0].view(1, -1).to(device)
    
    # Encode
    z_a = model_2d.encode(img_a)
    z_b = model_2d.encode(img_b)
    
    # Interpolate in latent space
    n_steps = 10
    alphas = np.linspace(0, 1, n_steps)
    
    interpolated_images = []
    for alpha in alphas:
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img_interp = model_2d.decode(z_interp)
        interpolated_images.append(img_interp.view(28, 28).cpu().numpy())
    
    # Visualize
    fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))
    for i, img in enumerate(interpolated_images):
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f'α={alphas[i]:.1f}', fontsize=9)
    
    plt.suptitle(f'Latent Space Interpolation: {class_names[class_a]} → {class_names[class_b]}', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. Convolutional Autoencoder

### 7.1 Why Convolutional Layers?

Fully-connected layers ignore spatial structure of images. **Convolutional autoencoders** are better for images because:

1. **Preserve spatial relationships**: Convolutions maintain 2D structure
2. **Parameter efficiency**: Fewer parameters than fully-connected
3. **Translation invariance**: Learn features regardless of position
4. **Better reconstruction quality**: Especially for complex images

### Architecture Design

**Encoder (Downsampling):**
```
Input (1×28×28) → Conv(16) → Conv(32) → Conv(64) → Flatten → Bottleneck
```

**Decoder (Upsampling):**
```
Bottleneck → Reshape → ConvTranspose(64) → ConvTranspose(32) → ConvTranspose(16) → Output (1×28×28)
```

In [ ]:
class ConvAutoencoder(nn.Module):
    """
    Convolutional Autoencoder for image reconstruction. Uses Conv2d for encoding and ConvTranspose2d for decoding.
    """
    def __init__(self, bottleneck_dim=128):
        super(ConvAutoencoder, self).__init__()
    
        # Encoder: Convolutional layers with downsampling
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # → 16×14×14
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),  # → 32×7×7
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # → 64×4×4
            nn.ReLU(True),
        )
        
        # Flatten and bottleneck
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(64 * 4 * 4, bottleneck_dim)
        
        # Bottleneck to spatial
        self.fc_decode = nn.Linear(bottleneck_dim, 64 * 4 * 4)
        self.unflatten = nn.Unflatten(1, (64, 4, 4))
        
        # Decoder: Transposed convolutions with upsampling
        self.decoder = nn.Sequential(
            # Input: 64×4×4
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=0),  # → 32×7×7
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # → 16×14×14
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),   # → 1×28×28
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # Encode
        x = self.encoder(x)
        x = self.flatten(x)
        x = self.fc_encode(x)
        
        # Decode
        x = self.fc_decode(x)
        x = self.unflatten(x)
        x = self.decoder(x)
        return x
    
    def encode(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        x = self.fc_encode(x)
        return x


# Initialize convolutional model
conv_model = ConvAutoencoder(bottleneck_dim=128).to(device)

total_params = sum(p.numel() for p in conv_model.parameters())
print(conv_model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Bottleneck dimension: 128")

### 7.2 Training Convolutional Autoencoder

We'll train on Fashion-MNIST for denoising.

In [ ]:
# Training configuration
num_epochs_conv = 20
conv_criterion = nn.MSELoss()
conv_optimizer = optim.Adam(conv_model.parameters(), lr=0.001)

conv_train_losses = []
conv_test_losses = []

print(f"Training Convolutional Autoencoder for {num_epochs_conv} epochs...")
start_time = time.time()

for epoch in range(num_epochs_conv):
    conv_model.train()
    train_loss = 0.0
    
    for images, _ in fashion_train_loader:
        images = images.to(device)
        
        # Add noise
        noisy_images = add_noise(images, noise_factor)
        
        # Forward pass
        reconstructed = conv_model(noisy_images)
        loss = conv_criterion(reconstructed, images)
        
        # Backward pass
        conv_optimizer.zero_grad()
        loss.backward()
        conv_optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(fashion_train_loader)
    conv_train_losses.append(avg_train_loss)
    
    # Evaluation
    conv_model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for images, _ in fashion_test_loader:
            images = images.to(device)
            noisy_images = add_noise(images, noise_factor)
            reconstructed = conv_model(noisy_images)
            loss = conv_criterion(reconstructed, images)
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(fashion_test_loader)
    conv_test_losses.append(avg_test_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_conv}] | "
              f"Train Loss: {avg_train_loss:.6f} | "
              f"Test Loss: {avg_test_loss:.6f}")

elapsed_time = time.time() - start_time
print(f"Training completed in {elapsed_time:.2f} seconds")

### 7.3 Convolutional Autoencoder Results

In [ ]:
# Plot training curves
plot_training_curves(conv_train_losses, conv_test_losses, 
                    "Convolutional Autoencoder - Training Progress")

In [ ]:
# Visualize reconstructions
conv_model.eval()

with torch.no_grad():
    conv_test_images, _ = next(iter(fashion_test_loader))
    conv_test_images = conv_test_images.to(device)
    
    conv_noisy_test = add_noise(conv_test_images, noise_factor)
    conv_reconstructed = conv_model(conv_noisy_test)
    
    visualize_reconstructions(conv_test_images, conv_noisy_test, conv_reconstructed, 
                            n=10, dataset_name="Fashion-MNIST (CNN)")

### 7.4 Comparison: Fully-Connected vs Convolutional

Let's compare the two architectures on the same test data.

In [ ]:
# Get reconstructions from both models
with torch.no_grad():
    # Fully-connected model
    fc_noisy_flat = conv_noisy_test.view(conv_noisy_test.size(0), -1)
    fc_reconstructed_flat = fashion_model(fc_noisy_flat)
    fc_reconstructed = fc_reconstructed_flat.view(-1, 1, 28, 28)
    
    # Convolutional model (already computed)
    # conv_reconstructed
    
    # Calculate metrics
    test_flat = conv_test_images.view(conv_test_images.size(0), -1)
    fc_recon_flat = fc_reconstructed.view(fc_reconstructed.size(0), -1)
    conv_recon_flat = conv_reconstructed.view(conv_reconstructed.size(0), -1)
    
    fc_metrics = calculate_metrics(test_flat, fc_recon_flat)
    conv_metrics = calculate_metrics(test_flat, conv_recon_flat)
    
    print("Architecture Comparison on Fashion-MNIST:")
    print(f"\nFully-Connected Autoencoder:")
    print(f"  Parameters: {sum(p.numel() for p in fashion_model.parameters()):,}")
    print(f"  MSE:  {fc_metrics['MSE']:.6f}")
    print(f"  PSNR: {fc_metrics['PSNR']:.2f} dB")
    print(f"\nConvolutional Autoencoder:")
    print(f"  Parameters: {sum(p.numel() for p in conv_model.parameters()):,}")
    print(f"  MSE:  {conv_metrics['MSE']:.6f}")
    print(f"  PSNR: {conv_metrics['PSNR']:.2f} dB")
    print(f"\nImprovement with CNN:")
    print(f"  MSE reduction: {(1 - conv_metrics['MSE']/fc_metrics['MSE'])*100:.2f}%")
    print(f"  PSNR gain: {conv_metrics['PSNR'] - fc_metrics['PSNR']:.2f} dB")

In [ ]:
# Visual comparison: Important Distinctions

fig, axes = plt.subplots(4, 10, figsize=(20, 8))

for i in range(10):
    # Original
    axes[0, i].imshow(conv_test_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    
    # Noisy
    axes[1, i].imshow(conv_noisy_test[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    
    # FC reconstruction
    axes[2, i].imshow(fc_reconstructed[i].cpu().squeeze(), cmap='gray')
    axes[2, i].axis('off')
    
    # Conv reconstruction
    axes[3, i].imshow(conv_reconstructed[i].cpu().squeeze(), cmap='gray')
    axes[3, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Noisy', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('FC Recon', fontsize=11, fontweight='bold')
axes[3, 0].set_ylabel('Conv Recon', fontsize=11, fontweight='bold')

plt.suptitle('Architecture Comparison: Fully-Connected vs Convolutional', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Experiments & Analysis

### 8.1 Effect of Bottleneck Size

The bottleneck dimension controls the compression level. Let's systematically explore its impact on reconstruction quality.

In [ ]:
# Test different bottleneck sizes
bottleneck_sizes = [2, 16]
bottleneck_results = []

print("Training autoencoders with different bottleneck sizes...")
for bottleneck_dim in bottleneck_sizes:
    print(f"\nBottleneck size: {bottleneck_dim}")
    
    # Create model
    test_model = Autoencoder(
        input_dim=784, 
        hidden_dims=[128, 64], 
        bottleneck_dim=bottleneck_dim).to(device)

    test_criterion = nn.MSELoss()
    test_optimizer = optim.Adam(test_model.parameters(), lr=0.001)
    
    # Train for fewer epochs (just for comparison)
    n_epochs = 10
    
    for epoch in range(n_epochs):
        test_model.train()
        for images, _ in fashion_train_loader:
            images = images.view(images.size(0), -1).to(device)
            noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_factor)
            noisy_images = noisy_images.view(noisy_images.size(0), -1)
            
            reconstructed = test_model(noisy_images)
            loss = test_criterion(reconstructed, images)
            
            test_optimizer.zero_grad()
            loss.backward()
            test_optimizer.step()
    
    # Evaluate
    test_model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for images, _ in fashion_test_loader:
            images = images.view(images.size(0), -1).to(device)
            noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_factor)
            noisy_images = noisy_images.view(noisy_images.size(0), -1)
            
            reconstructed = test_model(noisy_images)
            loss = test_criterion(reconstructed, images)
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(fashion_test_loader)
    compression_ratio = 784 / bottleneck_dim
    
    bottleneck_results.append({
        'size': bottleneck_dim,
        'loss': avg_test_loss,
        'compression': compression_ratio
    })
    
    print(f"  Test Loss: {avg_test_loss:.6f}")
    print(f"  Compression: {compression_ratio:.1f}x")

print("Experiment completed!")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes = [r['size'] for r in bottleneck_results]
losses = [r['loss'] for r in bottleneck_results]
compressions = [r['compression'] for r in bottleneck_results]

# Loss vs bottleneck size
axes[0].plot(sizes, losses, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Bottleneck Dimension', fontsize=12)
axes[0].set_ylabel('Test Loss (MSE)', fontsize=12)
axes[0].set_title('Reconstruction Quality vs Bottleneck Size', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# Loss vs compression ratio
axes[1].plot(compressions, losses, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Compression Ratio', fontsize=12)
axes[1].set_ylabel('Test Loss (MSE)', fontsize=12)
axes[1].set_title('Reconstruction Quality vs Compression', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis()  # Higher compression on left

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- Larger bottleneck → Better reconstruction (lower loss)")
print("- Smaller bottleneck → Higher compression but more information loss")
print("- Trade-off between compression and quality")
print("- Diminishing returns beyond certain bottleneck size")

### 8.2 Effect of Noise Level

How does the autoencoder perform with different noise intensities?

In [ ]:
# Test different noise levels
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5]
noise_results = []

# Use the trained Fashion-MNIST model
fashion_model.eval()

print("Testing different noise levels...")
with torch.no_grad():
    for noise_level in noise_levels:
        test_loss = 0.0
        
        for images, _ in fashion_test_loader:
            images = images.view(images.size(0), -1).to(device)
            noisy_images = add_noise(images.view(-1, 1, 28, 28), noise_level)
            noisy_images = noisy_images.view(noisy_images.size(0), -1)
            
            reconstructed = fashion_model(noisy_images)
            loss = fashion_criterion(reconstructed, images)
            test_loss += loss.item()
        
        avg_loss = test_loss / len(fashion_test_loader)
        noise_results.append({'noise': noise_level, 'loss': avg_loss})
        
        print(f"Noise σ={noise_level:.1f}: Test Loss = {avg_loss:.6f}")

In [ ]:
# Visualize noise level impact
plt.figure(figsize=(10, 6))

noise_vals = [r['noise'] for r in noise_results]
noise_losses = [r['loss'] for r in noise_results]

plt.plot(noise_vals, noise_losses, 'go-', linewidth=2, markersize=10)
plt.xlabel('Noise Level (σ)', fontsize=12)
plt.ylabel('Test Loss (MSE)', fontsize=12)
plt.title('Denoising Performance vs Noise Intensity', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axvline(x=0.3, color='r', linestyle='--', alpha=0.5, label='Training noise level')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Model performs best near training noise level (σ=0.3)")
print("- Performance degrades with higher noise")
print("- Some generalization to different noise levels")

### 8.3 Visual Comparison Across Noise Levels

In [ ]:
# Visualize reconstructions at different noise levels
fashion_model.eval()

# Get a single test image
test_img, _ = fashion_test_dataset[9] # 8-slipper , 24-Trouser, 9-shoes
test_img = test_img.unsqueeze(0).to(device)

fig, axes = plt.subplots(3, len(noise_levels), figsize=(15, 6))

with torch.no_grad():
    for i, noise_level in enumerate(noise_levels):
        # Add noise
        noisy = add_noise(test_img, noise_level)
        
        # Reconstruct
        noisy_flat = noisy.view(1, -1)
        recon_flat = fashion_model(noisy_flat)
        recon = recon_flat.view(1, 1, 28, 28)
        
        # Display
        axes[0, i].imshow(test_img.cpu().squeeze(), cmap='gray')
        axes[0, i].set_title(f'σ={noise_level:.1f}', fontsize=10)
        axes[0, i].axis('off')
        
        axes[1, i].imshow(noisy.cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        
        axes[2, i].imshow(recon.cpu().squeeze(), cmap='gray')
        axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Noisy', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Reconstructed', fontsize=11, fontweight='bold')

plt.suptitle('Denoising Performance Across Noise Levels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary & Key Takeaways

**1. Autoencoder Fundamentals**
- Autoencoders learn compressed representations through unsupervised learning
- The bottleneck forces the network to learn efficient encodings
- Reconstruction loss drives the learning process

**2. Architecture Insights**
- **Fully-Connected**: Simple, works well for small images, many parameters
- **Convolutional**: Better for images, preserves spatial structure, more efficient
- **Bottleneck size**: Critical trade-off between compression and quality

**3. Applications Demonstrated**
- **Image Denoising**: Remove noise while preserving structure
- **Dimensionality Reduction**: Compress to 2D for visualization
- **Feature Learning**: Unsupervised representation learning

**4. Key Findings**
- Fashion-MNIST is more challenging than MNIST (higher loss)
- CNN autoencoders outperform fully-connected for images
- Latent space shows meaningful clustering without labels
- Performance degrades gracefully with increased noise

**5. Choosing Bottleneck Size:**
- Smaller → More compression, faster inference, lower quality
- Larger → Better quality, more storage, slower inference
- Optimal size depends on application requirements

**6. Architecture Selection:**
- Use **fully-connected** for: small images, non-spatial data
- Use **convolutional** for: larger images, spatial data, efficiency

**7. Loss Function:**
- **MSE**: Good for continuous values, penalizes large errors
- **BCE**: Better when treating pixels as probabilities
- **Perceptual loss**: Advanced option for better visual quality

**8. Next Steps:**
1. Experiment with your own datasets
2. Try different architectures and hyperparameters
3. Explore Variational Autoencoders (VAE)
4. Apply to real-world problems in your domain